# Geometria dos adversários ao redor do portador — GNN x Set Transformer x CNN

## 1. Introdução e objetivo

A geometria a ser modelada é o **(x, y) dos jogadores adversários (time pressionante) em torno do
portador da bola, com o portador na origem (0, 0)** e reorientado para a frente do portador = +X.

O professor observou que **colapsar essa geometria 2D em um rótulo único** (k-means/regras) **perde
informação**. Aqui comparamos, no mesmo problema, três famílias de modelos que leem a configuração
2D sem esse colapso — **GNN** (message passing), **Set Transformer** (atenção) e **CNN** (imagem
rasterizada) — além dos baselines de rótulo 1D (regras, k-means) e de um **DeepSets** simples.

Por fim, combinamos por **stacking** o melhor modelo de geometria com uma **regressão logística de
contexto** (não-espacial) — sem injetar rótulo na regressão. Resposta = `recovered` (recuperação em
até 10 s). Tabelas salvas em `classify_analysis/`.

## 2. Configuração, dados e engenharia de features

In [ ]:
import json, os, math, warnings
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')
np.random.seed(42)

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve, silhouette_score
from sklearn.model_selection import train_test_split

try:
    import torch
    import torch.nn as nn
    HAS_TORCH = True
    torch.manual_seed(42)
except Exception as e:
    HAS_TORCH = False
    print('torch indisponivel:', e)

OUT = '../classify_analysis'
os.makedirs(OUT, exist_ok=True)
LOCAL_R, NEIGH_R, KMAX = 30.0, 10.0, 14
print('HAS_TORCH =', HAS_TORCH)

In [ ]:
DATA_DIR = Path('../StatsBomb_2/data')
if not DATA_DIR.exists():
    DATA_DIR = Path('C:/Users/Abraao Ideapad3/Documents/Projetos/COE609/Analise_pressao_1/StatsBomb_2/data')
COMP_ID, SEASON_ID, RECOVERY_WINDOW = 55, 43, 10.0

def load_json(p):
    with open(p, encoding='utf-8') as fh:
        return json.load(fh)

matches = load_json(DATA_DIR / 'matches' / str(COMP_ID) / (str(SEASON_ID) + '.json'))
match_ids = sorted(m['match_id'] for m in matches)
parts, frames = [], {}
for mid in match_ids:
    edf = pd.json_normalize(load_json(DATA_DIR / 'events' / (str(mid) + '.json')), sep='_')
    edf['match_id'] = mid; parts.append(edf)
    tp = DATA_DIR / 'three-sixty' / (str(mid) + '.json')
    if tp.exists():
        for fr in load_json(tp):
            frames[fr['event_uuid']] = fr
events = pd.concat(parts, ignore_index=True)
print('jogos:', len(match_ids), '| eventos:', len(events), '| frames:', len(frames))

In [ ]:
def ts_to_seconds(ts):
    hh, mm, ss = ts.split(':'); return int(hh) * 3600 + int(mm) * 60 + float(ss)
events['t'] = events['timestamp'].apply(ts_to_seconds)
pressures = events[events['type_name'] == 'Pressure'].copy()
rec_fail = events['ball_recovery_recovery_failure'].fillna(False) if 'ball_recovery_recovery_failure' in events.columns else pd.Series(False, index=events.index)
duel_out = events.get('duel_outcome_name', pd.Series('', index=events.index)).fillna('')
rec_mask = ((events['type_name'].eq('Ball Recovery') & ~rec_fail) | events['type_name'].eq('Interception') | (events['type_name'].eq('Duel') & duel_out.str.contains('Won|Success')))
rec = events.loc[rec_mask, ['match_id', 'period', 'team_id', 't']]
L = pressures[['id', 'match_id', 'period', 'team_id', 't']].rename(columns={'t': 'tp'}).sort_values('tp')
R = rec.rename(columns={'t': 'tr'}).sort_values('tr')
mm = pd.merge_asof(L, R, left_on='tp', right_on='tr', by=['match_id', 'period', 'team_id'], direction='forward', tolerance=RECOVERY_WINDOW)
mm['recovered'] = mm['tr'].notna().astype(int)
pressures = pressures.merge(mm[['id', 'recovered']], on='id', how='left')
pressures['recovered'] = pressures['recovered'].fillna(0).astype(int)
pressures = pressures.reset_index(drop=True)
# contexto (nao-espacial) para o stacking
xy = pressures['location'].apply(lambda v: v[:2] if isinstance(v, list) and len(v) >= 2 else [np.nan, np.nan])
pressures['x'] = xy.apply(lambda v: v[0]); pressures['y'] = xy.apply(lambda v: v[1])
pressures['dist_to_goal'] = np.sqrt((120 - pressures['x']) ** 2 + (40 - pressures['y']) ** 2)
pressures['zone'] = pd.cut(pressures['x'], [-0.1, 40, 80, 120.1], labels=['Def', 'Mid', 'Att']).astype(str)
pressures['duration'] = pressures['duration'].fillna(0.0)
print('pressoes:', len(pressures), '| taxa de recuperacao: %.3f' % pressures['recovered'].mean())

### 2.1 Primitivas da geometria de adversários

Portador = jogador do time pressionado (`teammate==False`) mais próximo da bola, colocado na origem
(0,0); adversários = `teammate==True`, reorientados para a frente do portador = +X. As primitivas
(densidade radial 5/10/15 m, direção e dispersão) alimentam apenas os **baselines 1D**.

In [ ]:
PRIM2 = ['n_adv_5m', 'n_adv_10m', 'n_adv_15m', 'nearest_adv_dist', 'adv_arc_coverage',
         'largest_free_angle', 'free_lane_forward', 'adv_centroid_dist', 'adv_centroid_angle', 'adv_spread']

def adv_primitives(row):
    keys = PRIM2 + ['carrier_x', 'carrier_y', 'has_frame']
    nan = {k: np.nan for k in keys}; nan['has_frame'] = 0
    fr = frames.get(row['id'])
    if fr is None or not isinstance(row['location'], list):
        return pd.Series(nan)
    ball = np.array(row['location'][:2], float)
    advs, sups = [], []
    for p in fr['freeze_frame']:
        xyp = np.array(p['location'], float)
        (advs if p.get('teammate') else sups).append(xyp)
    if len(sups) == 0:
        return pd.Series(nan)
    sups = np.array(sups)
    carrier = sups[int(np.argmin(np.linalg.norm(sups - ball, axis=1)))]
    res = {k: 0 for k in PRIM2}
    res['has_frame'] = 1; res['carrier_x'] = float(carrier[0]); res['carrier_y'] = float(carrier[1])
    if len(advs) == 0:
        res['nearest_adv_dist'] = np.nan; res['largest_free_angle'] = 360.0
        res['adv_centroid_dist'] = np.nan; res['adv_centroid_angle'] = np.nan; res['adv_spread'] = np.nan
        return pd.Series(res)
    advs = np.array(advs); d = advs - carrier
    adv_r = np.column_stack([-d[:, 0], d[:, 1]])
    dist = np.linalg.norm(adv_r, axis=1)
    for r in (5, 10, 15):
        res['n_adv_%dm' % r] = int((dist <= r).sum())
    res['nearest_adv_dist'] = float(dist.min())
    near = adv_r[dist <= 10]
    angs = sorted(((np.degrees(np.arctan2(near[:, 1], near[:, 0])) + 360) % 360).tolist()) if len(near) else []
    res['adv_arc_coverage'] = len(set(int(a // 45) for a in angs)) / 8.0
    if len(angs) == 0:
        res['largest_free_angle'] = 360.0; fm = 0.0
    elif len(angs) == 1:
        res['largest_free_angle'] = 360.0; fm = (angs[0] + 180) % 360
    else:
        gaps = [((angs[(i + 1) % len(angs)] - angs[i]) % 360, i) for i in range(len(angs))]
        gap, i = max(gaps); fm = (angs[i] + gap / 2) % 360; res['largest_free_angle'] = gap
    res['free_lane_forward'] = int(math.cos(math.radians(fm)) > 0)
    loc = adv_r[dist <= LOCAL_R]
    if len(loc):
        cen = loc.mean(0)
        res['adv_centroid_dist'] = float(np.linalg.norm(cen))
        res['adv_centroid_angle'] = float((np.degrees(np.arctan2(cen[1], cen[0])) + 360) % 360)
        res['adv_spread'] = float(np.linalg.norm(loc, axis=1).std()) if len(loc) > 1 else 0.0
    else:
        res['adv_centroid_dist'] = np.nan; res['adv_centroid_angle'] = np.nan; res['adv_spread'] = np.nan
    return pd.Series(res)

prim = pressures.apply(adv_primitives, axis=1)
pressures = pd.concat([pressures, prim], axis=1)
print('cobertura de freeze frame: %.3f' % pressures['has_frame'].mean())

### 2.2 Representações da geometria (para os modelos profundos)

A mesma nuvem de adversários (centrada no portador, frente=+X) é codificada de três formas:
**point set / grafo** (até 14 adversários mais próximos, com features `(x, y, dist)` escalados e
adjacência por proximidade ≤10 m) e **imagem raster** (mapa de calor 32×32, ±25 m, bumps
gaussianos). Mantemos só amostras com ≥1 adversário no raio de 30 m.

In [ ]:
G, HALF, SIG = 32, 25.0, 2.0
xs = np.linspace(-HALF, HALF, G); gx, gy = np.meshgrid(xs, xs)
Xp_l, A_l, Mk_l, IMG_l, y_l, vidx = [], [], [], [], [], []
for i, row in pressures.iterrows():
    if row['has_frame'] != 1 or not isinstance(row['location'], list):
        continue
    fr = frames.get(row['id']); carrier = np.array([row['carrier_x'], row['carrier_y']], float)
    pts = []
    for p in fr['freeze_frame']:
        if p.get('teammate'):
            xyp = np.array(p['location'], float)
            if np.linalg.norm(xyp - carrier) <= LOCAL_R:
                dd = xyp - carrier; pts.append([-dd[0], dd[1]])
    if len(pts) < 1:
        continue
    pts = np.array(pts, float)
    pts = pts[np.argsort(np.linalg.norm(pts, axis=1))[:KMAX]]
    k = len(pts); dist = np.linalg.norm(pts, axis=1)
    feat = np.zeros((KMAX, 3), np.float32)
    feat[:k] = np.column_stack([pts[:, 0] / 30.0, pts[:, 1] / 30.0, dist / 30.0])
    mask = np.zeros(KMAX, np.float32); mask[:k] = 1.0
    Dm = np.linalg.norm(pts[:, None, :] - pts[None, :, :], axis=2)
    a = (Dm <= NEIGH_R).astype(float); np.fill_diagonal(a, 1.0)
    dinv = 1.0 / np.sqrt(a.sum(1, keepdims=True)); a = a * dinv * dinv.T
    A = np.zeros((KMAX, KMAX), np.float32); A[:k, :k] = a
    img = np.zeros((G, G), np.float32)
    for (px, py) in pts:
        img += np.exp(-(((gx - px) ** 2 + (gy - py) ** 2) / (2 * SIG ** 2)))
    Xp_l.append(feat); A_l.append(A); Mk_l.append(mask); IMG_l.append(img[None]); y_l.append(row['recovered']); vidx.append(i)
Xp = np.array(Xp_l, np.float32); A = np.array(A_l, np.float32); Mk = np.array(Mk_l, np.float32)
IMG = np.array(IMG_l, np.float32); y = np.array(y_l, np.float32); vidx = np.array(vidx)
print('amostras (>=1 adversario):', len(y), '| Xp', Xp.shape, '| IMG', IMG.shape, '| taxa y %.3f' % y.mean())

# Parte 1 — Modelos da geometria

## 3. Baseline A — tipologia por regras (rótulo 1D)

In [ ]:
hf = pressures[pressures['has_frame'] == 1]
FREE_MED = float(hf['largest_free_angle'].median())

def classify_rule(r):
    if r['has_frame'] != 1:
        return 'Sem_frame'
    if r['n_adv_10m'] <= 1:
        return 'Livre'
    if r['n_adv_5m'] >= 2:
        return 'Cercado'
    if r['free_lane_forward'] == 1 and r['largest_free_angle'] >= FREE_MED:
        return 'Frente_livre'
    return 'Frente_bloqueada'

pressures['ctx_rule'] = pressures.apply(classify_rule, axis=1)
sub = pressures[pressures['has_frame'] == 1]

def wilson(k, n, z=1.96):
    if n == 0: return (np.nan, np.nan)
    p = k / n; d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d
    h = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return (max(0, c - h), min(1, c + h))

rows = []
for cl, g in sub.groupby('ctx_rule'):
    n = len(g); k = int(g['recovered'].sum()); lo, hi = wilson(k, n)
    rows.append(dict(classe=cl, n=n, taxa_recuperacao=k / n, ic_inf=lo, ic_sup=hi))
rules_summary = pd.DataFrame(rows).sort_values('taxa_recuperacao').reset_index(drop=True)
rules_summary.to_csv(os.path.join(OUT, 'rules_summary.csv'), index=False)
ct = pd.crosstab(sub['ctx_rule'], sub['recovered'])
chi2, pchi, dof, _ = stats.chi2_contingency(ct)
cramer = np.sqrt(chi2 / (ct.values.sum() * (min(ct.shape) - 1)))
pd.DataFrame([dict(chi2=chi2, p_valor=pchi, cramers_v=cramer, n_classes=ct.shape[0])]).to_csv(os.path.join(OUT, 'rules_test.csv'), index=False)
print('Qui-quadrado p=%.2g | Cramers V=%.3f' % (pchi, cramer))
print(rules_summary.round(3).to_string(index=False))

In [ ]:
glob = sub['recovered'].mean()
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].bar(rules_summary['classe'], rules_summary['n'], color='#1565c0')
ax[0].set_title('Distribuicao das classes (regras)'); ax[0].tick_params(axis='x', rotation=20)
yerr = [rules_summary['taxa_recuperacao'] - rules_summary['ic_inf'], rules_summary['ic_sup'] - rules_summary['taxa_recuperacao']]
ax[1].bar(rules_summary['classe'], rules_summary['taxa_recuperacao'] * 100, yerr=np.array(yerr) * 100, capsize=4, color='#2e7d32')
ax[1].axhline(glob * 100, color='red', ls='--'); ax[1].set_ylabel('% recuperacao')
ax[1].set_title('Recuperacao por classe (linha = media geral)'); ax[1].tick_params(axis='x', rotation=20)
plt.tight_layout(); plt.show()

In [ ]:
# Mini-diagramas: adversarios (vermelho) ao redor do portador na origem (estrela preta)
def draw_adv(row, ax):
    fr = frames.get(row['id']); carrier = np.array([row['carrier_x'], row['carrier_y']])
    for p in fr['freeze_frame']:
        if p.get('teammate'):
            dd = np.array(p['location'], float) - carrier
            if np.linalg.norm(dd) <= 30:
                ax.scatter(-dd[0], dd[1], c='#c62828', s=30)
    ax.scatter(0, 0, c='black', marker='*', s=180)
    ax.axhline(0, color='gray', lw=0.5); ax.axvline(0, color='gray', lw=0.5)
    ax.set_xlim(-25, 25); ax.set_ylim(-25, 25); ax.set_aspect('equal')
    ax.set_xticks([]); ax.set_yticks([])

classes = list(rules_summary['classe'])
fig, axes = plt.subplots(1, len(classes), figsize=(3.0 * len(classes), 3.2))
if len(classes) == 1: axes = [axes]
for ax, cl in zip(axes, classes):
    draw_adv(sub[sub['ctx_rule'] == cl].iloc[0], ax); ax.set_title(cl, fontsize=9)
fig.suptitle('Exemplo por classe — adversarios em torno do portador (frente = +X)', fontsize=10)
plt.tight_layout(); plt.show()

## 4. Baseline B — k-means (rótulo 1D), k variável

In [ ]:
PRIMV = pressures.loc[vidx, PRIM2].astype(float)
PRIMV = PRIMV.fillna(PRIMV.median()).to_numpy()
Z = StandardScaler().fit_transform(PRIMV)
ks = range(2, 9); ksel = []
for k in ks:
    kmf = KMeans(n_clusters=k, n_init=10, random_state=42).fit(Z)
    ksel.append(dict(k=k, inertia=kmf.inertia_, silhouette=silhouette_score(Z, kmf.labels_, sample_size=4000, random_state=42)))
ksel = pd.DataFrame(ksel); ksel.to_csv(os.path.join(OUT, 'kmeans_k_selection.csv'), index=False)
K_BEST = int(ksel.loc[ksel['silhouette'].idxmax(), 'k'])
km = KMeans(n_clusters=K_BEST, n_init=10, random_state=42).fit(Z); KMLAB = km.labels_
crows = []
for c in range(K_BEST):
    msk = KMLAB == c; n = int(msk.sum()); kk = int(y[msk].sum()); lo, hi = wilson(kk, n)
    crows.append(dict(cluster=c, n=n, taxa_recuperacao=kk / n, ic_inf=lo, ic_sup=hi))
kmeans_clusters = pd.DataFrame(crows); kmeans_clusters.to_csv(os.path.join(OUT, 'kmeans_clusters.csv'), index=False)
prof = pd.DataFrame(Z, columns=PRIM2); prof['cluster'] = KMLAB
prof_mean = prof.groupby('cluster').mean(); prof_mean.round(3).to_csv(os.path.join(OUT, 'kmeans_profile.csv'))

fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].plot(ksel['k'], ksel['inertia'], '-o'); ax[0].set_title('Cotovelo'); ax[0].set_xlabel('k')
ax[1].plot(ksel['k'], ksel['silhouette'], '-o', color='#ef6c00'); ax[1].set_title('Silhueta'); ax[1].set_xlabel('k')
ax[2].bar(range(K_BEST), kmeans_clusters['taxa_recuperacao'] * 100, color='#2e7d32')
ax[2].axhline(y.mean() * 100, color='red', ls='--'); ax[2].set_title('Recuperacao por cluster (k=%d)' % K_BEST)
ax[2].set_xlabel('cluster'); ax[2].set_ylabel('% recuperacao')
plt.tight_layout(); plt.show()
print('k escolhido:', K_BEST)

## 5. Modelos profundos da geometria 2D — GNN, Set Transformer, CNN, DeepSets

Todos preveem `P(recovered)` a partir da geometria de adversários. **GNN** (GCN denso, message
passing), **Set Transformer** (atenção sobre o conjunto), **CNN** (sobre a imagem raster) e
**DeepSets** (baseline de conjunto). Aumento por reflexão (espelhar y) no treino dos modelos de
ponto/imagem.

In [ ]:
if HAS_TORCH:
    class DeepSets(nn.Module):
        def __init__(s, f=3, h=32):
            super().__init__()
            s.phi = nn.Sequential(nn.Linear(f, h), nn.ReLU(), nn.Linear(h, h), nn.ReLU())
            s.rho = nn.Sequential(nn.Linear(2 * h, h), nn.ReLU(), nn.Dropout(0.2), nn.Linear(h, 1))
        def forward(s, X, M):
            h = s.phi(X) * M.unsqueeze(-1); m = M.unsqueeze(-1)
            mean = (h * m).sum(1) / m.sum(1).clamp(min=1)
            mx = h.masked_fill(m == 0, -1e9).max(1).values
            return s.rho(torch.cat([mean, mx], 1)).squeeze(-1)

    class DenseGCN(nn.Module):
        def __init__(s, f=3, h=32, L=2):
            super().__init__()
            s.lins = nn.ModuleList([nn.Linear(f if i == 0 else h, h) for i in range(L)])
            s.head = nn.Sequential(nn.Linear(2 * h, h), nn.ReLU(), nn.Dropout(0.2), nn.Linear(h, 1))
        def forward(s, X, Adj, M):
            H = X
            for lin in s.lins:
                H = torch.relu(torch.bmm(Adj, lin(H))) * M.unsqueeze(-1)
            m = M.unsqueeze(-1)
            mean = (H * m).sum(1) / m.sum(1).clamp(min=1)
            mx = H.masked_fill(m == 0, -1e9).max(1).values
            return s.head(torch.cat([mean, mx], 1)).squeeze(-1)

    class SetTransformer(nn.Module):
        def __init__(s, f=3, h=32, heads=4, L=2):
            super().__init__()
            s.inp = nn.Linear(f, h)
            s.att = nn.ModuleList([nn.MultiheadAttention(h, heads, batch_first=True) for _ in range(L)])
            s.ff = nn.ModuleList([nn.Sequential(nn.Linear(h, h), nn.ReLU(), nn.Linear(h, h)) for _ in range(L)])
            s.n1 = nn.ModuleList([nn.LayerNorm(h) for _ in range(L)])
            s.n2 = nn.ModuleList([nn.LayerNorm(h) for _ in range(L)])
            s.head = nn.Sequential(nn.Linear(h, h), nn.ReLU(), nn.Dropout(0.2), nn.Linear(h, 1))
        def forward(s, X, M):
            h = torch.relu(s.inp(X)); kpm = (M == 0)
            for att, ff, n1, n2 in zip(s.att, s.ff, s.n1, s.n2):
                a, _ = att(h, h, h, key_padding_mask=kpm)
                h = n1(h + a); h = n2(h + ff(h))
            m = M.unsqueeze(-1)
            pooled = (h * m).sum(1) / m.sum(1).clamp(min=1)
            return s.head(pooled).squeeze(-1)

    class SmallCNN(nn.Module):
        def __init__(s):
            super().__init__()
            s.c = nn.Sequential(nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                               nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.AdaptiveAvgPool2d(4))
            s.head = nn.Sequential(nn.Flatten(), nn.Linear(32 * 16, 64), nn.ReLU(), nn.Dropout(0.2), nn.Linear(64, 1))
        def forward(s, img):
            return s.head(s.c(img)).squeeze(-1)

    Xpt = torch.tensor(Xp); At = torch.tensor(A); Mt = torch.tensor(Mk); IMGt = torch.tensor(IMG); yt = torch.tensor(y)
    print('tensores prontos:', Xpt.shape, IMGt.shape)

In [ ]:
if HAS_TORCH:
    pos = np.arange(len(y))
    tr, tmp = train_test_split(pos, test_size=0.40, stratify=y, random_state=42)
    stack_i, test_i = train_test_split(tmp, test_size=0.50, stratify=y[tmp], random_state=42)
    tr2, val_i = train_test_split(tr, test_size=0.20, stratify=y[tr], random_state=42)

    def make_fwd(kind, model):
        def fwd(idx, aug):
            t = torch.as_tensor(idx, dtype=torch.long)
            if kind == 'cnn':
                ib = IMGt[t]
                if aug and np.random.rand() < 0.5:
                    ib = torch.flip(ib, dims=[2])
                return model(ib)
            xb = Xpt[t]
            if aug and np.random.rand() < 0.5:
                xb = xb.clone(); xb[..., 1] = -xb[..., 1]
            if kind == 'gcn':
                return model(xb, At[t], Mt[t])
            return model(xb, Mt[t])
        return fwd

    def train_model(kind, model, epochs=40, bs=256, lr=1e-3):
        fwd = make_fwd(kind, model)
        opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
        lossf = nn.BCEWithLogitsLoss(); best, best_state, hist = -1, None, []
        for ep in range(epochs):
            model.train(); perm = np.random.permutation(tr2)
            for sx in range(0, len(perm), bs):
                b = perm[sx:sx + bs]; opt.zero_grad()
                loss = lossf(fwd(b, True), yt[torch.as_tensor(b, dtype=torch.long)])
                loss.backward(); opt.step()
            model.eval()
            with torch.no_grad():
                auc = roc_auc_score(y[val_i], torch.sigmoid(fwd(val_i, False)).numpy())
            hist.append(auc)
            if auc > best:
                best = auc; best_state = {k: v.clone() for k, v in model.state_dict().items()}
        model.load_state_dict(best_state); return hist

    def predict(kind, model, idx):
        model.eval()
        with torch.no_grad():
            return torch.sigmoid(make_fwd(kind, model)(idx, False)).numpy()

    models = {'deepsets': DeepSets(), 'gcn': DenseGCN(), 'settransf': SetTransformer(), 'cnn': SmallCNN()}
    hists, geom_stack, geom_test, geom_auc = {}, {}, {}, {}
    for kind, mdl in models.items():
        hists[kind] = train_model(kind, mdl)
        geom_stack[kind] = predict(kind, mdl, stack_i); geom_test[kind] = predict(kind, mdl, test_i)
        geom_auc[kind] = roc_auc_score(y[test_i], geom_test[kind])
        print('  %-10s test AUC = %.3f' % (kind, geom_auc[kind]))

In [ ]:
if HAS_TORCH:
    def label_rate_pred(labels, tr_pos, te_pos):
        s = pd.Series(y[tr_pos]).groupby(labels[tr_pos]).mean(); g = y[tr_pos].mean()
        return np.array([s.get(l, g) for l in labels[te_pos]])
    RULEV = pressures.loc[vidx, 'ctx_rule'].to_numpy()
    auc_rule = roc_auc_score(y[test_i], label_rate_pred(RULEV, tr, test_i))
    auc_km = roc_auc_score(y[test_i], label_rate_pred(KMLAB, tr, test_i))

    fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
    names = ['regras 1D', 'k-means 1D', 'DeepSets', 'GNN', 'SetTransf', 'CNN']
    vals = [auc_rule, auc_km, geom_auc['deepsets'], geom_auc['gcn'], geom_auc['settransf'], geom_auc['cnn']]
    cols = ['#b0bec5', '#b0bec5', '#90a4ae', '#1565c0', '#6a1b9a', '#2e7d32']
    ax[0].bar(names, vals, color=cols); ax[0].set_ylim(0.5, max(vals) + 0.02); ax[0].set_ylabel('AUC (teste)')
    ax[0].axhline(0.5, color='k', lw=0.6); ax[0].set_title('Rotulo 1D vs modelos 2D'); ax[0].tick_params(axis='x', rotation=20)
    for i, v in enumerate(vals): ax[0].text(i, v + 0.001, '%.3f' % v, ha='center', fontsize=8)
    for nm, sc in [('regras', label_rate_pred(RULEV, tr, test_i)), ('k-means', label_rate_pred(KMLAB, tr, test_i)),
                   ('GNN', geom_test['gcn']), ('SetTransf', geom_test['settransf']), ('CNN', geom_test['cnn'])]:
        fpr, tpr, _ = roc_curve(y[test_i], sc); ax[1].plot(fpr, tpr, label=nm)
    ax[1].plot([0, 1], [0, 1], 'k--', lw=0.7); ax[1].legend(fontsize=8); ax[1].set_title('ROC (teste)')
    ax[1].set_xlabel('FPR'); ax[1].set_ylabel('TPR')
    plt.tight_layout(); plt.show()
    print('AUC 1D: regras %.3f | kmeans %.3f' % (auc_rule, auc_km))

# Parte 2 — Combinação por stacking

## 6. Logística de contexto + meta-modelo

Modelo de contexto (não-espacial: posição, minuto, duração, terço) combinado por stacking com o
**melhor** modelo de geometria. Meta-modelo treinado no conjunto STACK, avaliado no TESTE.

In [ ]:
if HAS_TORCH:
    def context_matrix(idx_labels):
        s = pressures.loc[idx_labels]; z = pd.get_dummies(s['zone'])
        for c in ['Def', 'Mid', 'Att']:
            if c not in z: z[c] = 0
        M_ = np.column_stack([s['x'], s['y'], s['dist_to_goal'], s['minute'], s['duration'], z['Def'], z['Mid'], z['Att']]).astype(float)
        return np.nan_to_num(M_, nan=0.0)
    CTX = context_matrix(vidx)
    scaler = StandardScaler().fit(CTX[tr])
    ctx_model = LogisticRegression(max_iter=1000).fit(scaler.transform(CTX[tr]), y[tr])
    p_ctx_stack = ctx_model.predict_proba(scaler.transform(CTX[stack_i]))[:, 1]
    p_ctx_test = ctx_model.predict_proba(scaler.transform(CTX[test_i]))[:, 1]
    auc_ctx = roc_auc_score(y[test_i], p_ctx_test)

    best_geom = max(geom_auc, key=lambda k: roc_auc_score(y[stack_i], geom_stack[k]))
    meta = LogisticRegression(max_iter=1000).fit(np.column_stack([p_ctx_stack, geom_stack[best_geom]]), y[stack_i])
    p_meta_test = meta.predict_proba(np.column_stack([p_ctx_test, geom_test[best_geom]]))[:, 1]
    auc_meta = roc_auc_score(y[test_i], p_meta_test)

    comp = [dict(modelo='regras_1d', auc_teste=auc_rule), dict(modelo='kmeans_1d', auc_teste=auc_km)]
    for k in ['deepsets', 'gcn', 'settransf', 'cnn']:
        comp.append(dict(modelo=k, auc_teste=geom_auc[k]))
    comp += [dict(modelo='contexto', auc_teste=auc_ctx), dict(modelo='stacking(contexto+%s)' % best_geom, auc_teste=auc_meta)]
    model_comparison = pd.DataFrame(comp)
    model_comparison.to_csv(os.path.join(OUT, 'model_comparison.csv'), index=False)
    pd.DataFrame([dict(coef_contexto=meta.coef_[0][0], coef_geometria=meta.coef_[0][1], intercepto=meta.intercept_[0], melhor_geometria=best_geom)]).to_csv(os.path.join(OUT, 'stacking_meta.csv'), index=False)
    pd.DataFrame([dict(n_pressoes=len(pressures), n_amostras=len(y), taxa_recuperacao=float(y.mean()), has_torch=True, n_train=len(tr2), n_val=len(val_i), n_stack=len(stack_i), n_test=len(test_i), kmax_adv=KMAX)]).to_csv(os.path.join(OUT, 'run_info.csv'), index=False)
    print(model_comparison.round(4).to_string(index=False))
    print('melhor geometria:', best_geom, '| meta coef [contexto, geometria]:', np.round(meta.coef_[0], 3))

In [ ]:
if HAS_TORCH:
    fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
    for nm, sc in [('Contexto', p_ctx_test), ('Geometria (%s)' % best_geom, geom_test[best_geom]), ('Stacking', p_meta_test)]:
        fpr, tpr, _ = roc_curve(y[test_i], sc); ax[0].plot(fpr, tpr, label='%s (AUC=%.3f)' % (nm, roc_auc_score(y[test_i], sc)))
    ax[0].plot([0, 1], [0, 1], 'k--', lw=0.7); ax[0].legend(); ax[0].set_title('ROC no teste'); ax[0].set_xlabel('FPR'); ax[0].set_ylabel('TPR')
    s1 = ax[1].scatter(p_ctx_test, geom_test[best_geom], c=y[test_i], cmap='coolwarm', s=10, alpha=0.5)
    ax[1].set_xlabel('P - contexto'); ax[1].set_ylabel('P - geometria (%s)' % best_geom)
    ax[1].set_title('Complementaridade'); fig.colorbar(s1, ax=ax[1], label='recovered')
    plt.tight_layout(); plt.show()

## 7. Conclusão

Leia `classify_analysis/model_comparison.csv` para a comparação final. Pontos a avaliar:

- **Perda de informação 2D→1D:** comparar a AUC dos rótulos 1D (regras, k-means) com a dos modelos
  2D (GNN, Set Transformer, CNN, DeepSets). Um ganho dos modelos 2D sustenta o argumento do
  professor; ganhos pequenos indicam que, *para este desfecho*, a geometria dos adversários carrega
  pouco sinal preditivo isolado.
- **Qual arquitetura 2D vence:** a CNN (imagem) e o DeepSets tendem a ir bem em nuvens esparsas; a
  GNN simples pode não superar baselines sem features de nó mais ricas — vale registrar.
- **Stacking:** se a AUC do stacking supera a do contexto e a da geometria isolados, os sinais são
  complementares.

**Limitações:** freeze frame só vê jogadores no quadro; bola ≈ posição do evento; modelos compactos
e um único split (k-fold daria estimativas mais robustas); AUCs nesta faixa (~0,5–0,6) indicam sinal
fraco — recuperar a bola depende de muito além da geometria instantânea dos adversários.